<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/01-foundations-workflow.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)

## **Problems, Paradigms, and Workflow**

Machine learning is not defined by one algorithm or one model family. It is a way of building systems whose behavior is estimated from data, tested on evidence that was not used during fitting, and improved through an iterative engineering process. A useful machine learning solution therefore contains more than a trained model: it also contains a precisely framed task, representative data, an objective, an evaluation protocol, and a plan for using and monitoring predictions.

This chapter establishes the vocabulary used throughout the rest of the machine learning series. It separates ideas that are often mixed together:

- A **learning paradigm** describes the feedback available during learning, such as labels, unlabeled examples, or rewards.
- An **input-output formulation** describes what the system must predict, such as a class, number, ranking, generated object, or action.
- A **model** specifies a family of possible functions or probability distributions.
- A **learning algorithm** selects a model from that family using data and an objective.
- A **workflow** connects the model to the real problem, deployment environment, and future data.

Keeping these layers separate makes model selection easier and prevents a common beginner mistake: choosing an impressive algorithm before defining what success means.

### **What Does Learning Mean in Machine Learning?**

A conventional program receives explicit rules written by a developer. A learning system instead receives examples or interaction experience and uses an algorithm to estimate rules that perform well beyond those observations. A practical definition connects three elements:

- **Task $T$:** what the system must do, such as classify an email or predict a delivery time.
- **Experience $E$:** the information available for improvement, such as labeled examples, unlabeled data, demonstrations, or rewards.
- **Performance measure $P$:** the quantity used to judge improvement, such as recall, mean absolute error, ranking quality, or cumulative reward.

For supervised learning, a dataset is commonly written as

$$
D = \{(\mathbf{x}_i, y_i)\}_{i=1}^{n},
$$

where $\mathbf{x}_i$ is the feature representation of example $i$, $y_i$ is its desired output, and $n$ is the number of examples. A parameterized model $f_{\theta}$ maps an input to a prediction:

$$
\hat{y}_i = f_{\theta}(\mathbf{x}_i).
$$

The learning algorithm searches for parameters $\theta$ that make predictions useful according to a loss function $\mathcal{L}$. The average loss on the observed training data is the **empirical risk**:

$$
\widehat{R}(\theta)
= \frac{1}{n}\sum_{i=1}^{n}
\mathcal{L}\left(f_{\theta}(\mathbf{x}_i), y_i\right).
$$

Minimizing this quantity is not the final goal. The model will be used on future examples drawn from an unknown real-world distribution $p_{\text{data}}(X,Y)$. The ideal quantity is therefore the **population risk**:

$$
R(\theta)
= \mathbb{E}_{(X,Y)\sim p_{\text{data}}}
\left[\mathcal{L}(f_{\theta}(X),Y)\right].
$$

We cannot calculate population risk exactly because we do not possess every future example. A model has learned successfully when evidence from held-out data suggests that low training risk transfers to low population risk. This ability is called **generalization**. Memorizing the training set can reduce empirical risk without learning a reusable relationship.

Consider a rent predictor. Memorizing that one observed apartment rented for `$650` per week does not reveal how floor area, location, age, and condition relate to rent. Learning means estimating a relationship that remains useful for a different apartment. The distinction between fitting observations and discovering a transferable pattern is the central tension of machine learning.

<details>
<summary><strong>Python: A Minimal Learning Loop from Scratch</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(7)

# Experience E: observed apartment sizes and noisy weekly rents.
floor_area = rng.uniform(35, 120, size=80)
weekly_rent = 180 + 5.2 * floor_area + rng.normal(0, 35, size=80)

# Hold out examples before learning. They approximate future experience.
split = 60
x_train, x_test = floor_area[:split], floor_area[split:]
y_train, y_test = weekly_rent[:split], weekly_rent[split:]

# Standardization gives both parameters a well-scaled optimization problem.
area_mean = x_train.mean()
area_std = x_train.std()
x_train_scaled = (x_train - area_mean) / area_std

# Model in standardized coordinates: y_hat = weight * x_scaled + bias.
weight, bias = 0.0, 0.0
learning_rate = 0.05

for step in range(1_000):
    # 1. Forward pass: use the current parameters to predict rent.
    predictions = weight * x_train_scaled + bias

    # 2. Objective: mean squared error measures prediction error.
    residuals = predictions - y_train
    training_loss = np.mean(residuals ** 2)

    # 3. Learning algorithm: differentiate the loss and update parameters.
    gradient_weight = 2 * np.mean(residuals * x_train_scaled)
    gradient_bias = 2 * np.mean(residuals)
    weight -= learning_rate * gradient_weight
    bias -= learning_rate * gradient_bias

# Performance P: evaluate on examples that did not update the parameters.
x_test_scaled = (x_test - area_mean) / area_std
test_predictions = weight * x_test_scaled + bias
test_mae = np.mean(np.abs(test_predictions - y_test))

# Convert the standardized rule back to the original floor-area units.
original_weight = weight / area_std
original_bias = bias - weight * area_mean / area_std
print(f"Learned rule: rent = {original_weight:.2f} * area + {original_bias:.2f}")
print(f"Held-out mean absolute error: ${test_mae:.2f} per week")
```

</details>

This small example already contains the full learning pattern: define a model, make predictions, measure error, update parameters using training data, and evaluate on untouched examples. More sophisticated methods change the model, loss, optimizer, or feedback, but not this basic logic.


### **Core Components of a Learning Problem**

A machine learning problem becomes precise only when its components fit together. A strong model cannot repair an invalid label, an unrepresentative dataset, or a metric that rewards the wrong behavior.

| Component | Question it answers | Apartment-rent example |
|---|---|---|
| Example | What is one unit of prediction? | One apartment at one listing time |
| Features $X$ | What information is available before prediction? | Area, suburb, bedrooms, age |
| Label $Y$ | What outcome should be learned? | Weekly rent after agreement |
| Hypothesis space $\mathcal{H}$ | Which relationships may the learner consider? | Linear functions, trees, neural networks |
| Loss $\mathcal{L}$ | How is one error penalized? | Absolute or squared rent error |
| Metric | How will usefulness be reported? | MAE overall and by suburb |
| Decision rule | How will a prediction change an action? | Suggest a listing-price range |

#### **Data, Features, and Labels**

An **example** is the unit presented to the learner. In tabular data it is usually a row; in other domains it may be an image, document, graph, audio segment, user session, or time window. The raw object is transformed into a feature vector $\mathbf{x}\in\mathcal{X}$. For $n$ examples with $d$ numerical features, the design matrix has shape

$$
X \in \mathbb{R}^{n\times d}.
$$

A feature is not simply any available column. It must be known at the moment the prediction is made and represented in a form the model can use. A patient's diagnosis recorded after a laboratory test cannot be used to predict whether the test should be ordered; doing so would leak future information into training.

A **label** or target is the desired output in supervised learning. Good labels should be:

- **Operationally meaningful:** predicting the label should support a real decision.
- **Observable with acceptable noise:** different annotators or measurement systems should not disagree unpredictably.
- **Available at the correct time:** delayed outcomes require careful temporal splitting and monitoring.
- **Aligned with the real objective:** clicks may be an easy-to-measure proxy for satisfaction, but optimizing clicks alone may promote low-quality content.

Data quality has several dimensions. **Validity** asks whether values are legal, **accuracy** asks whether they reflect reality, **coverage** asks whether important cases are present, and **representativeness** asks whether the training population resembles the deployment population. More rows do not compensate for systematic sampling bias.

#### **Hypothesis Space and Model**

A **hypothesis** is one candidate mapping from inputs to outputs. The **hypothesis space** $\mathcal{H}$ is the collection of candidates that the learner is allowed to choose from:

$$
\mathcal{H} = \{f_{\theta}: \theta\in\Theta\}.
$$

For simple linear regression, $f_{\theta}(\mathbf{x})=\mathbf{w}^{\top}\mathbf{x}+b$, so $\theta=(\mathbf{w},b)$. A decision tree defines a different hypothesis space made of recursive feature tests. A neural network defines a very large space through layers of nonlinear transformations.

**Parameters** are learned from training data, such as linear weights or tree split values. **Hyperparameters** configure the learning process or hypothesis space, such as tree depth, regularization strength, or number of neighbors. Hyperparameters must be chosen using validation evidence rather than the final test set.

A larger hypothesis space can express more relationships, but expressiveness alone is not always beneficial. If the space is too restricted, the learner underfits. If it is extremely flexible relative to the available evidence, it can fit accidental noise. Data quantity, model capacity, regularization, and domain assumptions must be considered together.

#### **Objective and Learning Algorithm**

The **loss function** converts the error on one example into a number. Squared error is common for regression, while cross-entropy is common for probabilistic classification. The **objective function** aggregates losses and may add regularization:

$$
J(\theta)
= \underbrace{\frac{1}{n}\sum_{i=1}^{n}
\mathcal{L}(f_{\theta}(\mathbf{x}_i),y_i)}_{\text{fit to observed data}}
+ \lambda\underbrace{\Omega(\theta)}_{\text{preference for simpler solutions}}.
$$

The coefficient $\lambda$ controls the trade-off. A small value prioritizes training fit; a large value more strongly penalizes complexity. The exact meaning of complexity depends on the model. It may be large coefficients, a deep tree, a rough function, or a representation that violates domain constraints.

A **learning algorithm** searches for a low-objective hypothesis. Gradient descent updates differentiable parameters, tree induction selects splits, nearest-neighbor methods largely store examples, and Bayesian inference updates a distribution over hypotheses. The objective says what counts as good; the algorithm says how to search for it.

![A supervised model is updated after its prediction is compared with the actual label.](assets/ml-training-feedback.png){fig-align="center" width="58%" fig-alt="A prediction is compared with the actual label and the resulting error is used to update the model."}

*Figure source: Google for Developers, [Supervised Learning](https://developers.google.com/machine-learning/intro-to-ml/supervised), CC BY 4.0.*

The diagram should not be interpreted as changing the model whenever one production prediction is wrong. During training, many examples contribute update signals. During ordinary inference, parameters are fixed unless the system explicitly supports online learning.

#### **Inference and Prediction**

**Training** estimates parameters from experience. **Inference** applies the trained model to new inputs. Once parameters $\hat{\theta}$ are fixed, prediction is

$$
\hat{y}=f_{\hat{\theta}}(\mathbf{x}_{\text{new}}).
$$

The raw model output is not always the final decision. A classifier may output a probability $P(Y=1\mid X=x)=0.72$, while a product rule converts that probability into an action using a threshold. Different costs can imply different thresholds: a medical triage system may tolerate more false alarms to avoid missing a serious condition.

Inference also creates engineering constraints that do not appear in the training objective. A highly accurate model may still be unusable if predictions take too long, require unavailable features, exceed memory limits, leak private information, or cannot be reproduced consistently between training and serving.

#### **Evaluation and Generalization**

Evaluation estimates how the complete modeling procedure behaves on unseen data. A standard supervised workflow separates data into:

- A **training set** used to fit model parameters.
- A **validation set** used to compare models, tune hyperparameters, and choose thresholds.
- A **test set** used once for a final estimate after modeling decisions are fixed.

The **generalization gap** compares unseen performance with training performance. For losses, one useful form is

$$
\text{generalization gap}
= \widehat{R}_{\text{test}}(\hat{\theta})
- \widehat{R}_{\text{train}}(\hat{\theta}).
$$

A large positive gap often indicates overfitting, leakage in the experimental process, or a distribution mismatch. A small gap is not sufficient by itself: both losses can be poor when the model underfits. Evaluation must also match deployment. Random splitting is inappropriate when future data differ from past data, users appear repeatedly, or related records can leak across sets.

Finally, model metrics and product outcomes are different. Accuracy measures agreement with labels; it does not directly measure user trust, revenue, safety, latency, or fairness. A responsible evaluation reports the model metric, uncertainty, important data slices, resource costs, and the downstream effect of acting on predictions.


### **Types of Machine Learning and Tasks**

Learning paradigms are best distinguished by the **source of the training signal**, not by the algorithm name. The same neural network architecture can be trained with labels, self-generated targets, human preferences, or rewards and therefore participate in different paradigms.

| Paradigm | Experience available | Central question | Typical result |
|---|---|---|---|
| Supervised | Inputs paired with desired outputs | Can a mapping from $X$ to $Y$ generalize? | Classifier, regressor, ranker |
| Unsupervised | Inputs without externally supplied targets | What structure or distribution is present in $X$? | Clusters, embeddings, density model |
| Semi-supervised | A small labeled set plus a larger unlabeled set | Can unlabeled structure improve a supervised task? | Improved classifier or regressor |
| Reinforcement | Actions followed by rewards and new states | Which behavior maximizes long-term reward? | Policy or value function |

#### **Supervised Learning**

Supervised learning estimates a relationship from labeled examples. Given $D=\{(\mathbf{x}_i,y_i)\}_{i=1}^{n}$, the learner uses the observed targets to penalize incorrect predictions. It is appropriate when the desired outcome can be defined and measured for enough representative cases.

The strongest practical advantage is direct evaluation: predictions can be compared with held-out labels. Its main costs are label acquisition, label noise, and the risk that the chosen target is only a weak proxy for the actual goal.

##### **Classification**

Classification predicts a value from a finite set of categories:

$$
f:\mathcal{X}\rightarrow\{1,2,\ldots,K\}.
$$

A binary classifier might identify fraudulent transactions; a multiclass classifier might recognize one of ten digits; a multilabel classifier might assign several topics to one article. Modern classifiers usually estimate class scores or probabilities first and then apply a decision rule. This separation matters because the same probability model can use different thresholds under different error costs.

Classification is useful when categories correspond to distinct actions or interpretations. Converting a naturally continuous outcome into arbitrary bins can discard information, so the categories should reflect the decision rather than mere convenience.

##### **Regression**

Regression predicts a continuous quantity:

$$
f:\mathcal{X}\rightarrow\mathbb{R}^{m}.
$$

Examples include delivery time, energy demand, temperature, or a vector of spatial coordinates. A point prediction gives one number, while probabilistic regression can also represent uncertainty, prediction intervals, quantiles, or an entire conditional distribution.

Regression and classification are not determined by whether the label is written as a number. Postal codes are numeric symbols but form categories, while a star rating may be treated as ordinal, continuous, or categorical depending on the decision. The meaning of the output and loss determines the formulation.

<details>
<summary><strong>Python: Classification and Regression Side by Side</strong></summary>

```python
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Classification: the target is a category.
X_class, y_class = load_breast_cancer(return_X_y=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_class, y_class, test_size=0.25, stratify=y_class, random_state=42
)
classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000))
classifier.fit(Xc_train, yc_train)
class_predictions = classifier.predict(Xc_test)
print("Classification accuracy:", accuracy_score(yc_test, class_predictions))

# Regression: the target is a continuous measurement.
X_reg, y_reg = load_diabetes(return_X_y=True)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)
regressor = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
regressor.fit(Xr_train, yr_train)
value_predictions = regressor.predict(Xr_test)
print("Regression MAE:", mean_absolute_error(yr_test, value_predictions))
```

</details>

#### **Unsupervised Learning**

Unsupervised learning receives examples $D=\{\mathbf{x}_i\}_{i=1}^{n}$ without externally supplied targets. The goal is not simply to "find hidden patterns." A useful method must define which structure matters through an objective and assumptions about similarity, geometry, sparsity, or probability.

Because there may be no unique ground truth, unsupervised results require careful interpretation. Two clusterings can both be mathematically valid but answer different questions. Domain validation and downstream usefulness are often more informative than a single internal score.

##### **Clustering**

Clustering assigns or associates examples with groups so that selected notions of within-group similarity are high and between-group similarity is low. K-means, for example, minimizes squared distance from each point to its assigned centroid:

$$
\min_{C_1,\ldots,C_K,\boldsymbol{\mu}_1,\ldots,\boldsymbol{\mu}_K}
\sum_{k=1}^{K}\sum_{\mathbf{x}_i\in C_k}
\|\mathbf{x}_i-\boldsymbol{\mu}_k\|_2^2.
$$

This objective favors roughly spherical groups under Euclidean distance. Density-based and probabilistic methods encode different ideas of a cluster. Applications include exploratory segmentation, anomaly discovery, document organization, and the construction of pseudo-labels. A cluster should not automatically be interpreted as a real demographic or causal category.

##### **Dimensionality Reduction**

Dimensionality reduction maps high-dimensional examples into a lower-dimensional representation $\mathbf{z}\in\mathbb{R}^{k}$, where $k<d$:

$$
g:\mathbb{R}^{d}\rightarrow\mathbb{R}^{k}.
$$

Principal component analysis preserves directions of maximum variance under a linear projection. Manifold methods try to preserve selected local or global geometry, while autoencoders learn a representation that supports reconstruction. Dimensionality reduction can compress data, remove redundancy, accelerate later models, or support visualization. A two-dimensional visualization is an interpretation tool, not proof that the original data naturally contain well-separated groups.

##### **Association Rule Learning**

Association rules describe co-occurrence patterns such as $A\Rightarrow B$. For transactions $T$, three common quantities are:

$$
\operatorname{support}(A\Rightarrow B)=P(A\cup B),
$$

$$
\operatorname{confidence}(A\Rightarrow B)=P(B\mid A),
$$

$$
\operatorname{lift}(A\Rightarrow B)=\frac{P(B\mid A)}{P(B)}.
$$

Support asks how often the joint pattern occurs. Confidence asks how often $B$ appears when $A$ appears. Lift compares that confidence with the background frequency of $B$; a lift greater than one indicates positive association. None of these measures proves that buying $A$ causes buying $B$.

<details>
<summary><strong>Python: Clustering Followed by Dimensionality Reduction</strong></summary>

```python
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, known_classes = load_wine(return_X_y=True)

# K-means is distance based, so scaling is part of the modeling assumption.
clustering_pipeline = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=3, n_init=20, random_state=42),
)
cluster_ids = clustering_pipeline.fit_predict(X)

# PCA creates a two-dimensional view for inspection only.
projection_pipeline = make_pipeline(StandardScaler(), PCA(n_components=2))
X_2d = projection_pipeline.fit_transform(X)

print("Projected shape:", X_2d.shape)
print("First ten cluster assignments:", cluster_ids[:10])
# known_classes are used only for later interpretation, not for fitting K-means.
```

</details>

#### **Semi-Supervised Learning**

Semi-supervised learning combines a labeled set

$$
D_L=\{(\mathbf{x}_i,y_i)\}_{i=1}^{\ell}
$$

with a usually larger unlabeled set

$$
D_U=\{\mathbf{x}_j\}_{j=\ell+1}^{n}.
$$

It is useful when raw data are inexpensive but expert labels are scarce, as in medical imaging, speech, remote sensing, and content moderation. Unlabeled data help only through assumptions. The **smoothness assumption** says nearby examples tend to have similar labels; the **cluster assumption** says decision boundaries should pass through low-density regions; the **manifold assumption** says relevant variation lies on a lower-dimensional structure.

Common methods include pseudo-labeling, consistency regularization, graph-based label propagation, and teacher-student models. The main danger is confirmation bias: a model can assign incorrect pseudo-labels and then become more confident in its own mistakes. Confidence filtering, class balancing, strong validation, and human review are important safeguards.

Self-supervised learning is related but distinct. It constructs supervisory targets from the data itself, such as predicting a masked token or a transformed view. It is often used to pretrain representations before supervised fine-tuning.

<details>
<summary><strong>Python: Label Propagation with Few Labeled Examples</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.semi_supervised import LabelSpreading

X, true_labels = make_moons(n_samples=400, noise=0.18, random_state=42)

rng = np.random.default_rng(42)
observed_labels = np.full_like(true_labels, fill_value=-1)

# Keep only ten labels from each class; -1 marks an unlabeled example.
for class_id in np.unique(true_labels):
    candidates = np.flatnonzero(true_labels == class_id)
    selected = rng.choice(candidates, size=10, replace=False)
    observed_labels[selected] = class_id

model = LabelSpreading(kernel="knn", n_neighbors=12, alpha=0.2)
model.fit(X, observed_labels)

print("Observed labels:", np.sum(observed_labels != -1))
print("Accuracy for demonstration:", accuracy_score(true_labels, model.transduction_))
# true_labels are used here to evaluate the tutorial, not by the learner.
```

</details>

#### **Reinforcement Learning**

Reinforcement learning studies sequential decisions whose consequences affect future observations and rewards. At time $t$, an agent observes a state $S_t$, chooses an action $A_t$, receives reward $R_{t+1}$, and moves to $S_{t+1}$. A policy $\pi(a\mid s)$ specifies how actions are selected.

![The reinforcement learning loop between an agent and its environment.](assets/rl-agent-environment.svg){fig-align="center" width="74%" fig-alt="An agent sends an action to an environment and receives a new state and reward in return."}

*Figure source: Martin Thoma, [Agent-environment diagram](https://commons.wikimedia.org/wiki/File:Agent-environment-diagram-rl.svg), CC0.*

The objective is usually the expected discounted return:

$$
G_t=\sum_{k=0}^{\infty}\gamma^k R_{t+k+1},
\qquad
\max_{\pi}\;\mathbb{E}_{\pi}[G_t],
$$

where $0\leq\gamma<1$ discounts distant rewards. A small $\gamma$ emphasizes immediate outcomes; a value near one gives future outcomes more influence. The agent must also balance **exploration**, which gathers information about uncertain actions, with **exploitation**, which uses the best action currently known.

Reinforcement learning is appropriate when actions change future states or data collection. Robotics, adaptive resource allocation, games, and some recommendation settings have this property. If each decision is independent and an immediate correct label is already available, ordinary supervised learning is usually simpler and safer.

<details>
<summary><strong>Python: Epsilon-Greedy Learning in a Multi-Armed Bandit</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(42)
true_reward_means = np.array([0.20, 0.45, 0.35])
estimated_values = np.zeros(3)
action_counts = np.zeros(3, dtype=int)
epsilon = 0.10

for step in range(2_000):
    # Explore with probability epsilon; otherwise exploit the current estimate.
    if rng.random() < epsilon:
        action = rng.integers(len(true_reward_means))
    else:
        action = int(np.argmax(estimated_values))

    # The environment returns a noisy reward for the chosen action.
    reward = rng.normal(true_reward_means[action], 0.15)
    action_counts[action] += 1

    # Incremental mean: update only the selected action's value estimate.
    estimated_values[action] += (
        reward - estimated_values[action]
    ) / action_counts[action]

print("Estimated action values:", estimated_values.round(3))
print("Actions selected:", action_counts)
print("Learned best action:", int(np.argmax(estimated_values)))
```

</details>

The four paradigms solve different information problems. Supervised learning needs target labels, unsupervised learning defines structure without them, semi-supervised learning borrows structure to reduce label requirements, and reinforcement learning must learn from consequences generated by its own actions.


### **Common Input-Output Formulations**

A paradigm describes the feedback used for learning; a formulation describes the required output. These axes are independent. Recommendation can be trained from supervised click labels, pairwise preferences, self-supervised interaction logs, or reinforcement rewards. Starting from the output prevents an algorithm name from silently defining the wrong problem.

#### **Classification and Regression**

Classification produces categories or probabilities over categories, while regression produces continuous values. Both can be single-output or multi-output. Structured inputs such as images, text, graphs, and sequences can still support either formulation after representation learning.

The choice should follow the action. Predicting an exact delivery time suggests regression; deciding whether to display a delay warning may suggest classification. If a small numerical change crosses a product threshold and triggers a very different action, training directly for the categories or utility may align the model more closely with the decision.

#### **Ranking and Recommendation**

Ranking predicts an ordering rather than an isolated label. A scoring model $s_{\theta}(q,d)$ assigns a score to candidate $d$ for query or user context $q$, and candidates are sorted by score:

$$
d_{(1)},\ldots,d_{(K)}
= \operatorname{TopK}_{d\in\mathcal{C}(q)} s_{\theta}(q,d).
$$

Pointwise methods predict a relevance value for each item. Pairwise methods learn that one item should outrank another. Listwise methods optimize properties of the entire result list. Recommendation adds user-item interactions, changing preferences, exposure bias, diversity, novelty, and business constraints.

<details>
<summary><strong>Python: Convert Predicted Relevance into a Top-K Ranking</strong></summary>

```python
import numpy as np

candidate_ids = np.array(["article-A", "article-B", "article-C", "article-D"])
predicted_relevance = np.array([0.61, 0.87, 0.42, 0.79])

# Ranking is created by sorting scores, not by applying a class threshold.
order = np.argsort(predicted_relevance)[::-1]
top_k = 3

for rank, index in enumerate(order[:top_k], start=1):
    print(rank, candidate_ids[index], predicted_relevance[index])
```

</details>

Ranking quality is evaluated near the positions users actually inspect, using measures such as precision@$K$, recall@$K$, mean reciprocal rank, and normalized discounted cumulative gain. Ordinary accuracy ignores ordering and is usually inappropriate.

#### **Density Estimation and Generation**

Density estimation models how examples are distributed. An unconditional model estimates $p_{\theta}(x)$; a conditional model estimates $p_{\theta}(y\mid x)$. Once a probability distribution is available, the system can score likelihoods, detect unusual examples, fill missing values, or sample new outputs.

A discriminative classifier focuses on a decision boundary or $p(y\mid x)$. A generative model tries to represent how observations or outputs could be produced. For instance, digit classification maps an image to a digit label, while digit generation samples a new image from a learned distribution. Conditional generation includes translation, summarization, image synthesis from text, and probabilistic forecasting.

Generation is not judged by one universal metric. Likelihood, fidelity, diversity, factuality, safety, latency, and human preference may conflict. The evaluation protocol must reflect the intended use rather than relying on a single average score.

#### **Sequential Decision Making**

Sequential decision making outputs an action or policy rather than a static prediction. The core distinction is intervention: a chosen action changes what happens next. A route planner changes the vehicle's future location; a tutoring system changes what the learner sees; a recommender changes the interactions from which later models learn.

This feedback introduces delayed rewards, partial observability, off-policy evaluation, and safety constraints. A predictive model may still be one component of the system, but accurate prediction does not automatically imply an optimal action.

| Formulation | Typical output | Common training signal | Example metrics |
|---|---|---|---|
| Classification | Class or class probability | Labeled categories | F1, log loss, AUROC |
| Regression | Number, quantile, or interval | Measured continuous targets | MAE, RMSE, coverage |
| Ranking | Ordered candidate list | Relevance, clicks, preferences | NDCG@K, MRR, recall@K |
| Density/generation | Probability or generated sample | Observations, paired outputs, preferences | Log likelihood, task and human measures |
| Sequential decision | Action or policy | Rewards and transitions | Expected return, regret, safety violations |

The table describes outputs, not isolated model families. Trees can classify, regress, or rank; neural networks can support every row; probabilistic models can be either discriminative or generative. Formulation should be decided before architecture.


### **Inductive Bias and Modeling Assumptions**

Finite data are compatible with many possible explanations. If ten observed points lie on a smooth curve, infinitely many functions still pass through them. A learner can predict unobserved cases only by preferring some explanations over others. This preference is called **inductive bias**.

Inductive bias is not the same as statistical bias in a bias-variance decomposition. It is the broader set of assumptions that makes generalization possible. Examples include:

- **Linearity:** effects combine approximately as weighted sums.
- **Smoothness:** nearby inputs tend to have similar outputs.
- **Locality:** nearby training examples are especially informative, as in nearest neighbors.
- **Sparsity:** only a small number of features or parameters should be active.
- **Invariance:** selected transformations should not change the prediction.
- **Compositionality:** complex patterns can be built from reusable simpler parts.
- **Causal or domain constraints:** selected relationships must obey known structure.

Bias enters through feature representation, model architecture, regularization, data augmentation, priors, optimization, and even early stopping. There is no assumption-free learning algorithm that is best for every possible data-generating process. A useful bias matches stable structure in the problem.

![Polynomial models illustrating underfitting, an appropriate fit, and overfitting.](assets/underfitting-overfitting.png){fig-align="center" width="100%" fig-alt="Three polynomial regression plots of degrees one, four, and fifteen show underfitting, a suitable fit, and overfitting."}

*Figure source: scikit-learn developers, [Underfitting vs. Overfitting](https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html), BSD-3-Clause.*

In the left panel, a linear hypothesis space is too restrictive and produces high approximation error. The middle panel has enough flexibility to capture the stable pattern. The right panel is flexible enough to follow training noise, so its predictions are unstable away from observed points. Model complexity is therefore relative to both the data and the hypothesis space; a large model can generalize well when it has appropriate structure, regularization, and evidence.

Regularized empirical risk minimization makes the preference explicit:

$$
\hat{\theta}
= \arg\min_{\theta}
\left[
\widehat{R}(\theta)+\lambda\Omega(\theta)
\right].
$$

Here $\widehat{R}$ rewards fit to observed examples and $\Omega$ encodes a preference, such as smaller coefficients or smoother functions. The validation process estimates whether the chosen trade-off transfers to unseen data.

<details>
<summary><strong>Python: Observe How Polynomial Degree Changes Generalization</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(0)

def true_function(x):
    return np.cos(1.5 * np.pi * x)

# Training points contain noise; test points approximate the underlying function.
x_train = np.sort(rng.random(30))
y_train = true_function(x_train) + rng.normal(0, 0.1, size=30)
x_test = np.linspace(0, 1, 500)
y_test = true_function(x_test)

for degree in [1, 4, 25]:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression(),
    )
    model.fit(x_train[:, None], y_train)

    train_mse = mean_squared_error(y_train, model.predict(x_train[:, None]))
    test_mse = mean_squared_error(y_test, model.predict(x_test[:, None]))
    print(f"degree={degree:2d} train_MSE={train_mse:.4f} test_MSE={test_mse:.4f}")
```

</details>

A lower training error should not automatically win. The comparison must use validation or cross-validation, and the final test set must remain untouched until the procedure is fixed. Chapter 4 develops this connection among capacity, empirical risk, generalization bounds, bias, and variance in detail.


### **The End-to-End Machine Learning Workflow**

Machine learning development is an iterative system-building process. Data collection changes the model; model errors reveal data problems; deployment changes user behavior and future data. A notebook that trains one estimator is therefore an experiment inside a larger lifecycle.

![The four iterative phases of an end-to-end machine learning workflow.](assets/ml-development-phases.png){fig-align="center" width="100%" fig-alt="An end-to-end workflow containing ideation and planning, experimentation, pipeline building, and productionization, with feedback loops between phases."}

*Figure source: Google for Developers, [ML Development Phases](https://developers.google.com/machine-learning/managing-ml-projects/phases), CC BY 4.0.*

The arrows returning to earlier phases are essential. If experiments are not viable, revisit the problem or data rather than endlessly tuning the model. If production exposes training-serving skew, repair the pipeline. If user needs change, redefine the target and success criteria.

#### **Problem Framing and Success Criteria**

Begin with the decision or outcome, not the algorithm. A useful problem statement identifies:

- **Prediction unit:** one user, transaction, document, device, or time interval.
- **Prediction time:** exactly when the system must produce an output.
- **Available information:** features known at that moment.
- **Target and horizon:** what will be predicted and how far ahead.
- **Action:** what a person or system will do with the output.
- **Error costs:** which mistakes matter and to whom.
- **Constraints:** latency, compute, privacy, fairness, safety, and interpretability.
- **Success criteria:** model metrics plus product or scientific outcomes.

Suppose a maintenance team wants to reduce unexpected machine failures. "Predict failure" is incomplete. A stronger formulation is: every hour, use sensor data available up to that time to estimate whether each machine will fail within seven days, so technicians can prioritize inspections. Recall matters because missed failures are expensive, but precision and inspection capacity also constrain the number of alerts.

Before using ML, check whether deterministic rules, process redesign, better instrumentation, or a database query solve the problem more directly. ML is most useful when the pattern is difficult to specify manually, enough representative experience exists, and predictions can change an action.

#### **Data Collection and Baseline Construction**

Data collection should follow the prediction definition. Record the feature timestamp, label timestamp, entity identifier, sampling process, missingness, and data lineage. Split data according to deployment: later time periods for forecasting, unseen users for personalization, or unseen sites for cross-location generalization.

Exploratory checks should cover distributions, missing values, duplicates, class balance, label reliability, subgroup coverage, and suspiciously predictive features. Preprocessing must be fitted only on training data. Otherwise information from validation or test examples can influence feature scaling, imputation, encoding, or selection.

Build a baseline before a complex model. A baseline may be a constant prediction, a simple heuristic, a linear model, or the current production system. It verifies that the dataset and metric are wired correctly and reveals whether additional complexity creates meaningful value.

#### **Training, Validation, and Error Analysis**

Training fits parameters on the training set. Validation chooses features, model families, hyperparameters, and thresholds. Test evaluation estimates the performance of the completed procedure. Repeatedly checking the test set turns it into another validation set and makes the reported result optimistic.

Model comparison should report uncertainty across folds or repeated runs where feasible. Average performance can hide important failure modes, so error analysis examines slices such as rare classes, time periods, data sources, device types, languages, or demographic groups. Inspecting individual false positives and false negatives often reveals label ambiguity, missing features, leakage, or a mismatch between the metric and real costs.

The next experiment should respond to observed evidence. If both training and validation performance are poor, improve features, optimization, or model capacity. If training is strong but validation is weak, investigate leakage, variance, distribution mismatch, and regularization. If offline metrics are good but product outcomes are weak, revisit framing and the action policy.

#### **Deployment, Monitoring, and Iteration**

Deployment connects a fixed artifact to a serving path. Batch inference scores many records on a schedule; online inference responds to requests under latency constraints; edge inference runs on a device with limited resources. The same feature definitions must be implemented consistently in training and serving.

Production monitoring has several layers:

- **System health:** latency, throughput, memory, errors, and availability.
- **Data health:** schema violations, missingness, range changes, and feature drift.
- **Model behavior:** score distributions, calibration, slice performance, and delayed ground-truth metrics.
- **Decision impact:** interventions, user outcomes, fairness, safety incidents, and feedback loops.

Drift does not always require retraining. A feature distribution can change without hurting decisions, while the relationship between features and labels can change even when marginal feature distributions appear stable. Alerts should connect to diagnosis, rollback, retraining, or human review procedures.

<details>
<summary><strong>Python: A Leakage-Aware Supervised Workflow with a Baseline</strong></summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Load a self-contained labeled dataset.
X, y = load_breast_cancer(return_X_y=True)

# 2. Hold out the test set before preprocessing or model selection.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42,
)

# 3. Establish a constant baseline using training labels only.
baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)
baseline_auc = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])

# 4. Put preprocessing inside the pipeline to prevent test-data leakage.
model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2_000)),
    ]
)
model.fit(X_train, y_train)

# 5. Evaluate probabilities and thresholded decisions separately.
positive_probability = model.predict_proba(X_test)[:, 1]
predictions = (positive_probability >= 0.50).astype(int)

print(f"Baseline ROC-AUC: {baseline_auc:.3f}")
print(f"Model ROC-AUC: {roc_auc_score(y_test, positive_probability):.3f}")
print(classification_report(y_test, predictions, digits=3))

# 6. In production, persist the entire pipeline, its data schema,
#    threshold, metric report, and training-data version together.
```

</details>

This example is intentionally simple. A real workflow would add cross-validation, hyperparameter selection, confidence intervals, slice analysis, artifact versioning, deployment tests, and monitoring. Simplicity here makes the boundaries visible: the baseline, preprocessing, model, threshold, and metric are separate decisions.


### **Baselines and Sanity Checks**

A baseline answers a more useful question than "is the score high?": **is the learned system better than a credible alternative under the same evaluation protocol?** Without a baseline, a 92% accuracy result may be meaningless if the majority class already accounts for 95% of examples.

Useful baselines form a ladder:

| Baseline | What it tests |
|---|---|
| Constant or random predictor | Whether the metric exceeds chance or class prevalence |
| Simple business rule | Whether ML improves on available domain knowledge |
| Linear or shallow model | Whether nonlinear complexity is justified |
| Previous production model | Whether replacing the current system creates value |
| Human or oracle estimate | How much room for useful improvement may remain |

Sanity checks test the experiment itself. They are especially important because leakage can produce excellent-looking results.

- **Random-label test:** shuffle training labels. Performance should fall near chance; otherwise leakage or evaluation reuse is likely.
- **Tiny-subset overfit test:** a sufficiently expressive implementation should fit a very small batch. Failure can expose optimization or coding errors.
- **Feature-removal test:** remove suspicious features, identifiers, or timestamps and observe whether performance collapses.
- **Pipeline isolation test:** fit imputers, scalers, vocabularies, and feature selectors on training data only.
- **Duplicate and entity-overlap test:** ensure related examples do not cross train and test boundaries.
- **Temporal backtest:** train on the past and evaluate on a later period when deployment is temporal.
- **Slice test:** compare performance for rare classes and important populations instead of relying only on the mean.

<details>
<summary><strong>Python: A Random-Label Leakage Sanity Check</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000))

real_score = cross_val_score(pipeline, X, y, cv=5, scoring="roc_auc").mean()

rng = np.random.default_rng(42)
shuffled_labels = rng.permutation(y)
shuffled_score = cross_val_score(
    pipeline, X, shuffled_labels, cv=5, scoring="roc_auc"
).mean()

print(f"Real-label ROC-AUC: {real_score:.3f}")
print(f"Shuffled-label ROC-AUC: {shuffled_score:.3f}")
# The shuffled score should be close to 0.5 for a binary ROC-AUC task.
```

</details>

A sanity check does not prove that a system is correct, but a failed check is strong evidence that the reported evaluation should not be trusted. Run these tests early, before expensive tuning makes a flawed pipeline look convincing.


### **Choosing a Learning Paradigm**

The correct starting point is the structure of the decision and available feedback. A practical sequence is:

1. Define the real-world outcome and the action that follows a prediction.
2. Identify the prediction unit, time, horizon, and information available at that time.
3. Determine whether reliable target labels or rewards exist.
4. Choose the output formulation: class, value, ranking, distribution, generated object, or action.
5. Specify error costs, constraints, and a credible non-ML baseline.
6. Select the simplest paradigm and model that can represent the required relationship.

| Situation | Likely starting point | Main caution |
|---|---|---|
| Reliable historical inputs and desired outputs | Supervised learning | Label leakage and proxy-target mismatch |
| No labels; goal is exploration or representation | Unsupervised learning | Patterns may not be unique or actionable |
| Few labels and many related unlabeled examples | Semi-supervised learning | Incorrect pseudo-labels can reinforce errors |
| Targets can be constructed from raw data | Self-supervised pretraining | Pretraining objective may not match downstream needs |
| Actions affect future states and rewards are delayed | Reinforcement learning | Exploration, off-policy evaluation, and safety |
| Need an ordered list for each context | Ranking or recommendation formulation | Exposure bias and position-dependent evaluation |
| Need new samples or structured outputs | Generative or conditional-density modeling | Fidelity, diversity, factuality, and safety trade-offs |

Machine learning may be the wrong tool when rules are stable and easy to specify, data do not represent the deployment setting, outcomes cannot be measured, errors are unacceptable without human control, or a prediction cannot change any decision. In those cases, deterministic software, statistics, causal experimentation, optimization, process redesign, or better data collection may solve the actual problem more directly.

Several examples illustrate the reasoning:

- **Spam filtering:** labeled messages and a discrete action suggest supervised classification. Probabilities allow thresholds to reflect the unequal cost of blocking legitimate mail.
- **Customer exploration:** no accepted segments and an exploratory goal suggest clustering, followed by domain validation rather than treating cluster IDs as truth.
- **Search results:** the desired output is an ordered list, so ranking metrics and exposure-aware data matter more than ordinary classification accuracy.
- **Robot navigation:** actions change future states and rewards, so sequential decision making is central. Supervised perception models may still support the policy.
- **Document generation:** the output is structured and open ended, so conditional generation is appropriate, with separate checks for usefulness, factuality, safety, and latency.

The main lesson is that **problem formulation precedes model selection**. Data determine what can be learned, assumptions determine how evidence is generalized, metrics determine what is rewarded, and deployment determines whether the prediction creates value. Later chapters deepen each component, but every successful machine learning project continues to rely on this foundation.
